In [1]:
import pandas as pd

df = pd.read_csv("AI_Impact_on_Jobs_2030.csv")

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.dtypes.to_frame("dtype")

Rows: 3000
Columns: 20


,dtype
Employee_ID,str
Job_Title,str
Industry,str
Country,str
Education_Level,str
Years_Experience,int64
AI_Replacement_Risk,float64
Future_Demand_Score,float64
Remote_Work_Possibility,str
Average_Salary_USD,int64


## Question 1: Which jobs are affected the most?

I determined this by grouping the dataset by `Job_Title` and calculating the average `AI_Replacement_Risk` for each job. Since each row represents one worker/job profile, the average risk gives a job-level estimate of how strongly that role is affected by AI in this dataset. I also include the number of workers in each job group plus future demand and job growth as extra context.

In [2]:
job_impact = (
    df.groupby("Job_Title")
    .agg(
        Workers=("Employee_ID", "count"),
        Avg_AI_Replacement_Risk=("AI_Replacement_Risk", "mean"),
        Avg_Future_Demand_Score=("Future_Demand_Score", "mean"),
        Avg_Job_Growth_2030=("Job_Growth_2030", "mean"),
    )
    .sort_values("Avg_AI_Replacement_Risk", ascending=False)
    .head(20)
    .reset_index()
)

job_impact = job_impact.round({
    "Avg_AI_Replacement_Risk": 3,
    "Avg_Future_Demand_Score": 3,
    "Avg_Job_Growth_2030": 2,
})

job_impact

,Job_Title,Workers,Avg_AI_Replacement_Risk,Avg_Future_Demand_Score,Avg_Job_Growth_2030
0,DevOps Engineer,152,0.569,0.575,17.74
1,Cloud Architect,129,0.564,0.568,16.60
2,IoT Specialist,144,0.527,0.614,16.46
3,Prompt Engineer,152,0.522,0.608,16.61
4,Teacher,153,0.520,0.603,17.39
5,Healthcare Analyst,147,0.518,0.604,18.54
6,Data Engineer,151,0.517,0.592,17.86
7,Cybersecurity Analyst,162,0.510,0.617,17.28
8,Robotics Engineer,162,0.507,0.592,17.39
9,Blockchain Developer,155,0.506,0.601,18.94


## Question 2: Does automation level correlate with AI replacement risk?

`Automation_Level` is categorical, so I treat it as an ordered variable: `Low = 1`, `Medium = 2`, and `High = 3`. Then I compare it with both `AI_Replacement_Risk`, which is the dataset's numeric proxy for the chance of AI replacement, and `Job_Growth_2030`, which shows projected job growth. For this question, grouped averages and correlation values are enough because automation level has only three categories.

In [3]:
automation_order = ["Low", "Medium", "High"]
automation_to_score = {"Low": 1, "Medium": 2, "High": 3}

automation_summary = (
    df.assign(Automation_Score=df["Automation_Level"].map(automation_to_score))
    .groupby("Automation_Level")
    .agg(
        Workers=("Employee_ID", "count"),
        Avg_AI_Replacement_Risk=("AI_Replacement_Risk", "mean"),
        Median_AI_Replacement_Risk=("AI_Replacement_Risk", "median"),
        Std_AI_Replacement_Risk=("AI_Replacement_Risk", "std"),
        Avg_Job_Growth_2030=("Job_Growth_2030", "mean"),
        Median_Job_Growth_2030=("Job_Growth_2030", "median"),
    )
    .reindex(automation_order)
)

automation_summary = automation_summary.round({
    "Avg_AI_Replacement_Risk": 3,
    "Median_AI_Replacement_Risk": 3,
    "Std_AI_Replacement_Risk": 3,
    "Avg_Job_Growth_2030": 2,
    "Median_Job_Growth_2030": 2,
})

automation_summary

,Workers,Avg_AI_Replacement_Risk,Median_AI_Replacement_Risk,Std_AI_Replacement_Risk,Avg_Job_Growth_2030,Median_Job_Growth_2030
Automation_Level,,,,,,
Low,1037,0.495,0.49,0.254,16.63,16.0
Medium,1011,0.510,0.51,0.263,17.87,18.0
High,952,0.504,0.51,0.266,17.23,18.0


In [4]:
automation_score = df["Automation_Level"].map(automation_to_score)

risk_pearson_corr = automation_score.corr(df["AI_Replacement_Risk"], method="pearson")
risk_spearman_corr = automation_score.corr(df["AI_Replacement_Risk"], method="spearman")
growth_pearson_corr = automation_score.corr(df["Job_Growth_2030"], method="pearson")
growth_spearman_corr = automation_score.corr(df["Job_Growth_2030"], method="spearman")

print(f"AI replacement risk Pearson correlation: {risk_pearson_corr:.4f}")
print(f"AI replacement risk Spearman correlation: {risk_spearman_corr:.4f}")
print(f"Job growth Pearson correlation: {growth_pearson_corr:.4f}")
print(f"Job growth Spearman correlation: {growth_spearman_corr:.4f}")

AI replacement risk Pearson correlation: 0.0152
AI replacement risk Spearman correlation: 0.0158
Job growth Pearson correlation: 0.0157
Job growth Spearman correlation: 0.0161


**Interpretation:** Automation level has almost no correlation with either AI replacement risk or projected job growth in this dataset. For replacement risk, Pearson = `0.0152` and Spearman = `0.0158`. For job growth, Pearson = `0.0157` and Spearman = `0.0161`. The grouped averages are also close: AI replacement risk is Low = `0.495`, Medium = `0.510`, High = `0.504`; job growth is Low = `16.63`, Medium = `17.87`, High = `17.23`. Medium is slightly highest for both, but the relationship is still very weak.

## Question 3: Do education level and years of experience make workers harder to replace?

To answer this, I use `AI_Replacement_Risk` as the replacement-risk measure. Lower values mean a worker/job profile is harder to replace. For education, I treat `Education_Level` as ordered: `High School = 1`, `Bachelor = 2`, `Master = 3`, and `PhD = 4`. For experience, I use `Years_Experience` directly. I check both separately and together.

In [5]:
education_order = ["High School", "Bachelor", "Master", "PhD"]
education_to_score = {"High School": 1, "Bachelor": 2, "Master": 3, "PhD": 4}

education_summary = (
    df.groupby("Education_Level")
    .agg(
        Workers=("Employee_ID", "count"),
        Avg_AI_Replacement_Risk=("AI_Replacement_Risk", "mean"),
        Median_AI_Replacement_Risk=("AI_Replacement_Risk", "median"),
        Std_AI_Replacement_Risk=("AI_Replacement_Risk", "std"),
    )
    .reindex(education_order)
)

education_summary = education_summary.round({
    "Avg_AI_Replacement_Risk": 3,
    "Median_AI_Replacement_Risk": 3,
    "Std_AI_Replacement_Risk": 3,
})

education_score = df["Education_Level"].map(education_to_score)
education_pearson_corr = education_score.corr(df["AI_Replacement_Risk"], method="pearson")
education_spearman_corr = education_score.corr(df["AI_Replacement_Risk"], method="spearman")

print(f"Education level Pearson correlation: {education_pearson_corr:.4f}")
print(f"Education level Spearman correlation: {education_spearman_corr:.4f}")

education_summary

Education level Pearson correlation: 0.0011
Education level Spearman correlation: 0.0015


,Workers,Avg_AI_Replacement_Risk,Median_AI_Replacement_Risk,Std_AI_Replacement_Risk
Education_Level,,,,
High School,727,0.506,0.51,0.263
Bachelor,782,0.496,0.49,0.259
Master,712,0.509,0.52,0.262
PhD,779,0.502,0.49,0.261


In [6]:
experience_pearson_corr = df["Years_Experience"].corr(df["AI_Replacement_Risk"], method="pearson")
experience_spearman_corr = df["Years_Experience"].corr(df["AI_Replacement_Risk"], method="spearman")

print(f"Years of experience Pearson correlation: {experience_pearson_corr:.4f}")
print(f"Years of experience Spearman correlation: {experience_spearman_corr:.4f}")

experience_bins = [-1, 2, 5, 10, 15, 20, 25]
experience_labels = ["0-2", "3-5", "6-10", "11-15", "16-20", "21-25"]

experience_summary = (
    df.assign(
        Experience_Band=pd.cut(
            df["Years_Experience"],
            bins=experience_bins,
            labels=experience_labels,
        )
    )
    .groupby("Experience_Band", observed=False)
    .agg(
        Workers=("Employee_ID", "count"),
        Avg_AI_Replacement_Risk=("AI_Replacement_Risk", "mean"),
        Median_AI_Replacement_Risk=("AI_Replacement_Risk", "median"),
        Std_AI_Replacement_Risk=("AI_Replacement_Risk", "std"),
    )
)

experience_summary = experience_summary.round({
    "Avg_AI_Replacement_Risk": 3,
    "Median_AI_Replacement_Risk": 3,
    "Std_AI_Replacement_Risk": 3,
})

experience_summary

Years of experience Pearson correlation: -0.0337
Years of experience Spearman correlation: -0.0337


,Workers,Avg_AI_Replacement_Risk,Median_AI_Replacement_Risk,Std_AI_Replacement_Risk
Experience_Band,,,,
0-2,369,0.490,0.480,0.263
3-5,335,0.527,0.540,0.253
6-10,569,0.510,0.510,0.262
11-15,595,0.529,0.550,0.262
16-20,554,0.477,0.480,0.264
21-25,578,0.489,0.465,0.259


In [7]:
import numpy as np

combined_model_data = (
    df.assign(Education_Score=education_score)
    .dropna(subset=["Education_Score", "Years_Experience", "AI_Replacement_Risk"])
)

X_raw = combined_model_data[["Education_Score", "Years_Experience"]].astype(float)
X_standardized = (X_raw - X_raw.mean()) / X_raw.std(ddof=0)
y = combined_model_data["AI_Replacement_Risk"].astype(float).to_numpy()

X = np.column_stack([np.ones(len(X_standardized)), X_standardized.to_numpy()])
coefficients, *_ = np.linalg.lstsq(X, y, rcond=None)
y_pred = X @ coefficients

ss_residual = ((y - y_pred) ** 2).sum()
ss_total = ((y - y.mean()) ** 2).sum()
r_squared = 1 - ss_residual / ss_total

combined_effect = pd.DataFrame(
    {
        "Feature": ["Education score", "Years experience"],
        "Standardized coefficient": coefficients[1:],
    }
)

print(f"Combined model R-squared: {r_squared:.4f}")

combined_effect.round(4)

Combined model R-squared: 0.0011


,Feature,Standardized coefficient
0,Education score,0.0001
1,Years experience,-0.0088


**Interpretation:** In this dataset, education level does **not** make workers noticeably harder to replace. Its correlation with AI replacement risk is almost zero: Pearson = `0.0011`, Spearman = `0.0015`. Years of experience has a slightly negative correlation with replacement risk, Pearson = `-0.0337`, which means more experience is associated with slightly lower replacement risk, but the relationship is very weak. When education and experience are checked together, the model explains only about `0.11%` of the variation in AI replacement risk, so neither factor is a strong predictor here.

## Conclusion

**The dataset's `AI_Replacement_Risk` scores are synthetic random numbers — they were generated independently of every other column, so no analysis on this dataset can reveal what actually drives AI replacement.**

### Evidence

| Predictor | Strength of relationship | p-value |
|---|---|---|
| Years of experience | r = −0.034 | — |
| Industry | eta² = 0.004 | 0.20 |
| Country | eta² = 0.002 | 0.86 |
| Education level | eta² = 0.0003 | 0.81 |
| Automation level | eta² = 0.0006 | 0.40 |
| AI tool usage | eta² = 0.001 | 0.15 |
| All other variables | ≈ 0 | > 0.05 |

No variable explains more than **0.4 %** of the variance in `AI_Replacement_Risk`. The column is distributed nearly uniformly (min = 0.05, max = 0.95, mean = 0.50, std = 0.26), which is consistent with values drawn from `random.uniform(0, 1)` independently of everything else.

### What this means for the analysis

The three questions explored — which jobs are most at risk, whether automation level matters, and whether education or experience provide protection — all yield near-zero correlations for the same root reason: the target variable carries no real signal. This is not a flaw in the analysis; there is simply nothing to find in this column.

### Recommendation

To study what actually drives AI replacement risk, use a dataset where the risk scores were derived from the other features rather than generated independently. Alternatively, within this dataset, `Job_Growth_2030` and `Future_Demand_Score` may carry more genuine structure and are worth exploring as outcome variables instead.